# Kestrel V3, step 1: new SFT mix

Two runtimes, one notebook. `Run all` does whichever stage is next, and every stage resumes.

| stage | runtime | what | cost |
|---|---|---|---|
| A | **CPU** | download and tokenize the new mix (`scripts/prepare_sft_v3.py`), save to Drive | ~0.1 unit |
| B | **A100** | retrain SFT from the pre-SFT 4k checkpoint (`runs/ctx4k`), about 2 epochs | ~4 units |
| C | **A100** | same fixed chat eval on V1 and on the new model (`scripts/eval_chat.py`) | ~2 units |

Stage A stops the notebook when it's done, so nothing runs on the wrong card. Switch the
runtime to A100 and `Run all` again.


In [ ]:
# 0. Which runtime?
import os, json, shutil, glob, subprocess
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '|', GPU or 'CPU runtime')
if GPU and 'A100' not in GPU and 'H100' not in GPU:
    raise SystemExit(f'Got {GPU}. Use a CPU runtime for stage A, or an A100 for stages B and C.')


In [ ]:
# 1. Drive (signed in as the account that owns Kestrel/)
from google.colab import drive
drive.mount('/content/drive')
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
SFT3 = f'{KD}/datasets/sft_v3'
RUN = f'{KD}/runs/sft_v3'
CTX = f'{KD}/runs/ctx4k/ckpt.pt'
os.makedirs(f'{KD}/evals', exist_ok=True)
print('Drive OK:', sorted(os.listdir(KD)))
print('ctx4k checkpoint:', os.path.exists(CTX), '| new SFT set on Drive:', os.path.exists(f'{SFT3}/manifest.json'))


In [ ]:
# 2. Code (public repo)
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers datasets


In [ ]:
# A. Build the SFT set (CPU runtime). manifest.json is written last, so its presence
#    on Drive means the set is complete.
if not os.path.exists(f'{SFT3}/manifest.json'):
    if GPU:
        raise SystemExit('The SFT set is not on Drive yet. Stage A is CPU work: switch to a CPU runtime.')
    !python -u -m scripts.prepare_sft_v3 --examples 25000 --seq 4096 --out /content/data_sft_v3
    assert os.path.exists('/content/data_sft_v3/manifest.json'), 'build failed - see the log above'
    os.makedirs(SFT3, exist_ok=True)
    for f in ['train_tokens.bin', 'train_mask.bin', 'val_tokens.bin', 'val_mask.bin', 'samples.jsonl', 'manifest.json']:
        shutil.copy(f'/content/data_sft_v3/{f}', f'{SFT3}/{f}')
    drive.flush_and_unmount()
m = json.load(open(f'{SFT3}/manifest.json')) if os.path.ismount('/content/drive') else json.load(open('/content/data_sft_v3/manifest.json'))
print(json.dumps({k: m[k] for k in ('examples', 'tokens', 'length', 'per_source', 'shortfall', 'short_prompt_audit')}, indent=1))
if not GPU:
    raise SystemExit('STAGE_A_DONE. Stage A is finished. Switch the runtime to A100 and Run all for stages B and C.')


In [ ]:
# B. SFT from the pre-SFT 4k checkpoint, NOT from V1: V1's refusal habit shouldn't be the
#    starting point. Same LRs and tokens/step as V1 (196,608), about 2 epochs of this set.
#    batch 16 x accum 3 is the step-0 result. Resumes from Drive after a disconnect.
assert os.path.exists(CTX), f'{CTX} is missing'
sync = '/content/data_sft_v3'
os.makedirs(sync, exist_ok=True)
for f in glob.glob(f'{SFT3}/*'):
    out = f'{sync}/{os.path.basename(f)}'
    if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
        shutil.copy(f, out)
os.makedirs(RUN, exist_ok=True)
if not os.path.exists(f'{RUN}/ckpt.pt'):
    print('seeding from ctx4k ...'); shutil.copy(CTX, f'{RUN}/ckpt.pt')
CTX_STEP = 19590        # where the ctx4k run ended
n_steps = round(2 * m['tokens']['train'] / 196608)
final = CTX_STEP + n_steps
print(f"{m['tokens']['train']/1e6:.1f}M train tokens -> {n_steps} steps, ending at step {final}")
def finished():
    p = f'{RUN}/status.json'
    return os.path.exists(p) and json.load(open(p)).get('state') == 'done'
if not finished():
    !python -u -m kestrel.train --preset nano --sft-dir {sync} \
        --bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 16 --accum 3 \
        --steps {final} --warmup 15 --decay-frac 0.5 --muon-lr 0.004 --adamw-lr 0.0006 \
        --rope-scale 4 --out {RUN} --log-every 5 --eval-every 25 --eval-iters 10 \
        --ckpt-every 50 --ckpt-secs 1800
assert finished(), 'SFT did not finish - see the log above. Run all again to resume.'
print('STAGE_B_DONE')


In [ ]:
# C. The same questions to V1 (the release weights) and to the new model.
OUT = f'{KD}/evals/chat_eval_v3.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
V1 = '/content/kestrel-nano-v1-weights.pt'
if 'V1' not in done:
    if not os.path.exists(V1):
        !wget -q -O {V1} https://github.com/YoMosa2009/Kestrel/releases/download/v1.0.0/kestrel-nano-v1-weights.pt
    !python -u -m scripts.eval_chat --ckpt {V1} --label V1 --out {OUT}
if 'step1' not in done:
    !python -u -m scripts.eval_chat --ckpt {RUN}/ckpt.pt --label step1 --out {OUT}
runs = json.load(open(OUT))
for k, r in runs.items():
    print(k, json.dumps(r['summary']))
print('STAGE_C_DONE')
